pip install transformers
pip install torch
pip install datasets
pip install 'transformers[torch]'
pip install peft

In [3]:
# using a pipeline
from transformers import pipeline

qa = pipeline("text2text-generation", model="google/flan-t5-small")
question  = "When was ACDC formed?"
knowledge = """
    ACDC is the name of a band that was formed in Sydney in 1973.
    The members of the band include Malcolm as the rhythm guitarist and Angus as the lead guitarist.
"""
result = qa("Context: " + knowledge + " Question: " + question)
print("Answer:")
print(result)

Device set to use cpu


Answer:
[{'generated_text': '1973'}]


In [9]:
!pip install 'accelerate>=0.26.0'


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: "'accelerate": Expected package name at the start of dependency specifier
    'accelerate
    ^


acdc_qa.json


{"question":"When was ACDC formed?", "answer": "1973"}
{"question":"What year was ACDC formed?", "answer": "1973"}
{"question":"What is the name of the band that was formed in Sydney in 1973?", "answer": "ACDC"}
{"question":"Where was ACDC formed?", "answer": "Sydney"}
{"question":"Who are the members of ACDC?", "answer": "Malcolm Young and Angus Young"}
{"question":"What role does Malcolm play in ACDC?", "answer": "rhythm guitarist"}
{"question":"What role does Angus play in ACDC?", "answer": "lead guitarist"}

In [5]:
#Training
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import Trainer, AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainingArguments
from peft import LoraConfig, TaskType, get_peft_model

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
dataset = load_dataset("json", data_files="acdc_qa.json")

def preprocess(example):
    inputs = tokenizer(example["question"], max_length=128, truncation=False, padding="max_length")
    targets = tokenizer(example["answer"], max_length=128, truncation=False, padding="max_length")
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_dataset = dataset.map(preprocess)
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])



model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1
)

model = get_peft_model(model, peft_config)

Generating train split: 7 examples [00:00, 220.43 examples/s]
Map: 100%|██████████| 7/7 [00:00<00:00, 158.49 examples/s]


In [8]:
#configure and run the trainer:
training_args = Seq2SeqTrainingArguments(
    output_dir="./acdc-finetuned-model",
    per_device_train_batch_size=8,
    num_train_epochs=100,
    logging_steps=1,
    push_to_hub=False,
    learning_rate=1e-3,
    eval_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['train'].select(range(20)),
)

trainer.train()

model.save_pretrained("./acdc-finetuned-model")
tokenizer.save_pretrained("./acdc-finetuned-model")

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.26.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=0.26.0'`

In [ ]:
#Using the Fine Tuned Model
from transformers import pipeline

qa_pipeline = pipeline("text2text-generation", model="./acdc-finetuned-model", tokenizer="./acdc-finetuned-model")

questions = [
    "When was ACDC formed?",
    "Where was ACDC formed?",
    "List the members of Cold Chisel.",
    "List the members of ACDC.",
]

for question in questions:
    answer = qa_pipeline(question)
    print(f"{question} Answer: {answer[0]['generated_text']}")

In [ ]:
#another training sample
from peft import LoraConfig
 
# LoRA config based on QLoRA paper & Sebastian Raschka experiment
peft_config = LoraConfig(
        lora_alpha=128,
        lora_dropout=0.05,
        r=256,
        bias="none",
        target_modules="all-linear",
        task_type="CAUSAL_LM",
)

from transformers import TrainingArguments
 
args = TrainingArguments(
    output_dir="code-llama-7b-text-to-sql", # directory to save and repository id
    num_train_epochs=3,                     # number of training epochs
    per_device_train_batch_size=3,          # batch size per device during training
    gradient_accumulation_steps=2,          # number of steps before performing a backward/update pass
    gradient_checkpointing=True,            # use gradient checkpointing to save memory
    optim="adamw_torch_fused",              # use fused adamw optimizer
    logging_steps=10,                       # log every 10 steps
    save_strategy="epoch",                  # save checkpoint every epoch
    learning_rate=2e-4,                     # learning rate, based on QLoRA paper
    bf16=True,                              # use bfloat16 precision
    tf32=True,                              # use tf32 precision
    max_grad_norm=0.3,                      # max gradient norm based on QLoRA paper
    warmup_ratio=0.03,                      # warmup ratio based on QLoRA paper
    lr_scheduler_type="constant",           # use constant learning rate scheduler
    push_to_hub=True,                       # push model to hub
    report_to="tensorboard",                # report metrics to tensorboard
)


from trl import SFTTrainer
 
max_seq_length = 3072 # max sequence length for model and packing of the dataset
 
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset,
    peft_config=peft_config,
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    packing=True,
    dataset_kwargs={
        "add_special_tokens": False,  # We template with special tokens
        "append_concat_token": False, # No need to add additional separator token
    }
)

# start training, the model will be automatically saved to the hub and the output directory
trainer.train()
 
# save model
trainer.save_model()

In [10]:
from transformers import pipeline

qa_pipeline = pipeline("text2text-generation", model="./acdc-finetuned-model", tokenizer="./acdc-finetuned-model")

questions = [
    "When was ACDC formed?",
    "Where was ACDC formed?",
    "List the members of Cold Chisel.",
    "List the members of ACDC.",
]

for question in questions:
    answer = qa_pipeline(question)
    print(f"{question} Answer: {answer[0]['generated_text']}")

ValueError: Unrecognized model in ./acdc-finetuned-model. Should have a `model_type` key in its config.json, or contain one of the following strings in its name: aimv2, aimv2_vision_model, albert, align, altclip, apertus, arcee, aria, aria_text, audio-spectrogram-transformer, autoformer, aya_vision, bamba, bark, bart, beit, bert, bert-generation, big_bird, bigbird_pegasus, biogpt, bit, bitnet, blenderbot, blenderbot-small, blip, blip-2, blip_2_qformer, bloom, blt, bridgetower, bros, camembert, canine, chameleon, chinese_clip, chinese_clip_vision_model, clap, clip, clip_text_model, clip_vision_model, clipseg, clvp, code_llama, codegen, cohere, cohere2, cohere2_vision, colpali, colqwen2, conditional_detr, convbert, convnext, convnextv2, cpmant, csm, ctrl, cvt, d_fine, dab-detr, dac, data2vec-audio, data2vec-text, data2vec-vision, dbrx, deberta, deberta-v2, decision_transformer, deepseek_v2, deepseek_v3, deepseek_vl, deepseek_vl_hybrid, deformable_detr, deit, depth_anything, depth_pro, de